# Session 4: LLM Orchestration Paradigms

This notebook is a comprehensive guide to understanding and comparing different paradigms for **LLM Orchestration** and **Tool Use**. We will implement a shared task—**Currency Conversion and Tip Calculation**—across five different approaches to compare their ergonomics, loop capabilities, and fit.

## The Shared Task
The agent must answer queries like: *"How much is $150 USD in EUR including a 15% tip?"*
To solve this, the agent must:
1. Convert the currency using a exchange-rate tool.
2. Calculate the tip on the converted currency using a calculation tool.
3. Return the final structured answer.

## Notebook Layout
* **Part 1**: LangChain Fundamentals
* **Part 2**: Beyond LangChain: Orchestration Alternatives (Raw Loops, LangGraph, CrewAI, DSPy)
* **Part 3**: Comparison, Metrics, and Takeaways
* **Part 4**: Exercises (Conceptual, Hands-on, and Open-ended)

## Setup & Installation

First, install the necessary libraries and configure the environment variables.

In [ ]:
# !pip install -qU langchain langchain-google-genai langgraph crewai dspy-ai python-dotenv pandas pydantic google-genai

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if "GOOGLE_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ImportError:
        pass

## Shared Helper Tools

Let's write the raw Python functions that will serve as our tools throughout the notebook.

In [ ]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> float:
    """Convert an amount from one currency to another using fixed exchange rates."""
    rates = {
        ("USD", "INR"): 83.0,
        ("INR", "USD"): 1.0 / 83.0,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.0 / 0.92,
        ("EUR", "INR"): 90.0,
        ("INR", "EUR"): 1.0 / 90.0
    }
    key = (from_currency.upper(), to_currency.upper())
    rate = rates.get(key, 1.0)
    return round(amount * rate, 2)

def calculate_total(amount: float, tip_percent: float) -> float:
    """Calculate the total amount including a tip percentage."""
    tip = amount * (tip_percent / 100)
    return round(amount + tip, 2)

# Part 1 — LangChain Fundamentals

Let's cover the foundational building blocks of LangChain orchestrators.

### 1. Model Setup & Basic Invoke

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
response = llm.invoke("Say hello in exactly one word.")
print(response.content)

### 2. Prompt Templates

Using `PromptTemplate` and `ChatPromptTemplate` to structure inputs dynamically.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Legacy PromptTemplate
template = PromptTemplate.from_template("What is the capital of {country}?")
print(template.format(country="France"))

# ChatPromptTemplate
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "What is the capital of {country}?")
])
print(chat_prompt.format(country="Germany"))

### 3. Chains via LCEL & Output Parsers

LangChain Expression Language (LCEL) allows you to chain components using the `|` operator, which overrides Python's `__or__` method to create runnables.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Prompt | LLM | Parser
chain = chat_prompt | llm | StrOutputParser()
print(chain.invoke({"country": "Japan"}))

### 4. Pydantic Output Parser

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class CapitalInfo(BaseModel):
    capital: str = Field(description="Name of the capital city")
    population_millions: float = Field(description="Estimated population in millions")

parser = PydanticOutputParser(pydantic_object=CapitalInfo)

pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the query. {format_instructions}"),
    ("human", "Provide capital info for {country}.")
])

pydantic_chain = pydantic_prompt | llm | parser

result = pydantic_chain.invoke({
    "country": "Spain",
    "format_instructions": parser.get_format_instructions()
})
print(type(result))
print(result)

### 5. Memory: Legacy vs. Modern

Legacy `ConversationBufferMemory` is deprecated in newer versions of LangChain in favor of `RunnableWithMessageHistory` to support stateless, session-oriented execution.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

history_store = {}

def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "Remember user details."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

memory_chain = RunnableWithMessageHistory(
    memory_prompt | llm | StrOutputParser(),
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "test_session"}}
print(memory_chain.invoke({"input": "My favorite fruit is Mango"}, config=config))
print(memory_chain.invoke({"input": "What is my favorite fruit?"}, config=config))

### 6. Tools and Agents

Let's define the LangChain tools and set up the classic `create_tool_calling_agent` + `AgentExecutor` loop.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor

@tool
def convert_currency_tool(amount: float, from_currency: str, to_currency: str) -> float:
    """Convert an amount from one currency to another using fixed exchange rates."""
    return convert_currency(amount, from_currency, to_currency)

@tool
def calculate_total_tool(amount: float, tip_percent: float) -> float:
    """Calculate the total amount including a tip percentage."""
    return calculate_total(amount, tip_percent)

tools = [convert_currency_tool, calculate_total_tool]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools to calculate exchange rates and totals."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

query = "How much is $150 USD in EUR with a 15% tip?"
result = executor.invoke({"input": query, "chat_history": []})
print(f"Agent Result: {result['output']}")

### 7. Retriever/RAG Chain Ecosystem Breadth

A quick look at LangChain's vector storage integration to highlight ecosystem breadth.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Minimal local vector database setup
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = InMemoryVectorStore.from_documents(
    [Document(page_content="The exchange rates are static and set for learning.")],
    embeddings
)
retriever = vectorstore.as_retriever()
print("Retrieved:", retriever.invoke("rates"))

# Part 2 — Beyond LangChain: Orchestration Alternatives

Let's compare four alternative approaches to solve the shared task.

## 1. Raw Manual Loop (Gemini Native SDK)

We use the direct `google-genai` SDK and hand-roll a loop that catches tool requests, calls the functions, and sends the values back.

In [ ]:
import time
from google import genai
from google.genai import types

def run_raw_manual_loop(query_text):
    client = genai.Client()
    tools_list = [convert_currency, calculate_total]
    
    messages = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=query_text)]
        )
    ]
    
    config = types.GenerateContentConfig(
        tools=tools_list,
        temperature=0.0
    )
    
    call_count = 0
    start_time = time.time()
    
    while True:
        call_count += 1
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config=config
        )
        
        # Check if model wants to call tools
        if response.function_calls:
            # Append model request
            messages.append(response.candidates[0].content)
            
            response_parts = []
            for function_call in response.function_calls:
                name = function_call.name
                args = function_call.args
                
                if name == "convert_currency":
                    result = convert_currency(args["amount"], args["from_currency"], args["to_currency"])
                elif name == "calculate_total":
                    result = calculate_total(args["amount"], args["tip_percent"])
                else:
                    result = "Error: Tool not found"
                
                response_parts.append(
                    types.Part.from_function_response(
                        name=name,
                        response={"result": result}
                    )
                )
            
            messages.append(
                types.Content(
                    role="tool",
                    parts=response_parts
                )
            )
        else:
            duration = time.time() - start_time
            return response.text, call_count, duration

text_res, calls, dur = run_raw_manual_loop(query)
print(f"Raw Loop Response: {text_res}\nCalls: {calls}, Duration: {dur:.2f}s")

## 2. LangGraph

LangGraph models agent execution as a stateful graph where actions are nodes and routing paths are edges. This naturally supports cycles and branching, which LCEL pipelines struggle to represent.

In [ ]:
from langgraph.prebuilt import create_react_agent

def run_langgraph_agent(query_text):
    start_time = time.time()
    agent_graph = create_react_agent(llm, tools=tools)
    
    # Run the graph synchronously
    response = agent_graph.invoke({"messages": [("user", query_text)]})
    
    duration = time.time() - start_time
    # Calculate call count (input + tools execution iterations + output summary)
    calls = len([m for m in response["messages"] if m.type in ["ai", "human"]])
    return response["messages"][-1].content, calls, duration

graph_res, graph_calls, graph_dur = run_langgraph_agent(query)
print(f"LangGraph Response: {graph_res}\nCalls: {graph_calls}, Duration: {graph_dur:.2f}s")

## 3. CrewAI

CrewAI uses role-based, multi-agent execution where specialized agents perform tasks and delegate back and forth.

In [ ]:
from crewai import Agent, Task, Crew, Process

def run_crewai_agent(query_text):
    start_time = time.time()
    
    # Define Agents
    converter_agent = Agent(
        role="Currency Converter",
        goal="Accurately convert currency values using the tool",
        backstory="Expert in exchange rates and financial conversions.",
        tools=[convert_currency_tool],
        llm=llm,
        verbose=False
    )
    
    calculator_agent = Agent(
        role="Tip Calculator",
        goal="Calculate total amounts with tip using the tool",
        backstory="Expert in math computations and invoice checks.",
        tools=[calculate_total_tool],
        llm=llm,
        verbose=False
    )
    
    # Define Tasks
    task_convert = Task(
        description="Convert the starting amount to the target currency. Request: {input}",
        expected_output="The converted amount value.",
        agent=converter_agent
    )
    
    task_calculate = Task(
        description="Calculate the final total including the requested tip percentage on the converted amount.",
        expected_output="The final amount value including tip.",
        agent=calculator_agent
    )
    
    crew = Crew(
        agents=[converter_agent, calculator_agent],
        tasks=[task_convert, task_calculate],
        process=Process.sequential
    )
    
    result = crew.kickoff(inputs={"input": query_text})
    duration = time.time() - start_time
    return str(result), 2, duration # 2 sequential tasks

# Note: Multi-agent setups have higher startup latency
crew_res, crew_calls, crew_dur = run_crewai_agent(query)
print(f"CrewAI Response: {crew_res}\nDuration: {crew_dur:.2f}s")

## 4. DSPy (Declarative Self-improving Language Programs)

DSPy models task compilation using signatures. **Crucial Note:** DSPy is designed to solve *prompt and weight optimization* (programming the model), rather than *state/step orchestration* (looping tools). It compiles signature predictions.

In [ ]:
import dspy

class CurrencyAndTipSignature(dspy.Signature):
    """Convert starting currency to target currency and compute total including tip."""
    query = dspy.InputField(desc="The original prompt containing amount, target currency, and tip percentage")
    final_answer = dspy.OutputField(desc="The final converted value and total amount with tip")

def run_dspy_program(query_text):
    start_time = time.time()
    
    # Connect dspy to Gemini via GenAI model connector
    gemini_lm = dspy.Google("gemini-1.5-flash", api_key=os.environ["GOOGLE_API_KEY"])
    dspy.settings.configure(lm=gemini_lm)
    
    predictor = dspy.Predict(CurrencyAndTipSignature)
    response = predictor(query=query_text)
    
    duration = time.time() - start_time
    return response.final_answer, 1, duration

dspy_res, dspy_calls, dspy_dur = run_dspy_program(query)
print(f"DSPy Response: {dspy_res}\nDuration: {dspy_dur:.2f}s")

# Part 3 — Comparison & Wrap-up

### Side-by-Side Comparison

| Approach | Loop / Cycle Support | Boilerplate Level | Debuggability | Best-fit Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Raw Manual Loop** | Full (Native `while` loop) | High (Parse calls manually) | Simple (Print statements) | Low-overhead, single-file scripts |
| **LangChain LCEL** | Poor (Requires branching hooks)| Medium | Hard (Pipes stack trace) | Linear chain sequences & quick APIs |
| **LangGraph** | Excellent (Node routing edges) | Medium-High | Medium (Graph visualizers) | Stateful, looping multi-turn agents |
| **CrewAI** | Structured (Agent handoff loops) | High | Complex | Hierarchical document writing/roleplay |
| **DSPy** | N/A (Linear modules) | Low | Medium | Prompt optimization & zero-shot compilation |

### Practitioner Takeaways
1. **LCEL** is fantastic for simple parsing and linear API flows, but rapidly degrades in ergonomics when tools must be retried or state must loop.
2. **Raw manual loops** are extremely easy to inspect, debug, and trace, but require massive boilerplate as your tools and parameters scale.
3. **LangGraph** is the production choice when loops are a core feature of your workflow (like write-run-compile cycles).
4. **CrewAI** adds too much initialization latency for quick math/lookup tasks but thrives in complex document creation or role-based assignments.
5. **DSPy** solves prompt engineering, not tool loop control. Use it to optimize prompts *within* your LangGraph nodes.

### Shared Task Execution Metrics

Let's print the actual metrics recorded during local execution.

In [ ]:
import pandas as pd

metrics = {
    "Approach": ["Raw Manual Loop", "LangGraph", "CrewAI", "DSPy"],
    "LLM Call Count": [calls, graph_calls, crew_calls, dspy_calls],
    "Wall-Clock Time (s)": [f"{dur:.2f}", f"{graph_dur:.2f}", f"{crew_dur:.2f}", f"{dspy_dur:.2f}"]
}
print(pd.DataFrame(metrics))

# Part 4 — Exercises

## Conceptual (Answer in Markdown)

### Question 1
Why does an LCEL chain (`prompt | llm | parser`) struggle with a workflow that needs to loop back and retry a tool call? What specifically breaks?

*Your Answer Here*

### Question 2
When would CrewAI's multi-agent delegation actually outperform a single agent with multiple tools, rather than just adding latency?

*Your Answer Here*

### Question 3
DSPy and LangGraph could be combined in one pipeline. Describe where each would sit.

*Your Answer Here*

## Hands-on

### Question 4
Add a third tool, `get_historical_rate(currency, days_ago)`, and wire it into the raw manual loop. Compare the change required vs. adding it to the LangChain agent.

In [ ]:
# Write your code here

### Question 5
Force a failure: make `calculate_total` raise an error on malformed input (e.g. tip < 0), add retry logic to the raw loop version, then do the same in LangGraph. Compare how much code each required.

In [ ]:
# Write your code here

### Question 6
Extend the LangGraph agent so it asks the user for confirmation before returning a final answer over $500 (human-in-the-loop checkpoint) — show why LCEL chains can't easily do this.

In [ ]:
# Write your code here

## Open-ended & Stretch

### Question 7
Sketch (markdown only, no need to implement) a real workflow from your own work — e.g. an insurance claims triage step — and which of the five approaches you'd choose and why.

*Your Answer Here*

### Question 8
Run the shared task through all five approaches, log wall-clock time and number of LLM calls for each, and produce a results table. Which was fastest? Which used the fewest tokens?

*Your Answer Here*

### Question 9 (Stretch)
Swap DSPy's Predict for a `dspy.MIPROv2` optimizer with 3-5 labeled examples of currency/tip tasks. Does the optimized prompt outperform the hand-written one?

*Your Answer Here*